# QSS 45 Final Project — Final Figures
**Author:** Balla Sy

This notebook loads the cleaned QSS 45 dataset and model outputs from the project folders, then creates the final presentation-ready figures and descriptive table.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")

data = pd.read_csv(DATA_DIR / "qss45_bronx_manhattan_clean.csv")
model_comparison = pd.read_csv(OUTPUT_DIR / "model_comparison.csv")
full_ols_results = pd.read_csv(OUTPUT_DIR / "full_ols_results.csv")
vif_table = pd.read_csv(OUTPUT_DIR / "vif_table.csv")

print("Data rows:", len(data))
print("Final-figure notebook ready.")


## 1. Commute-time distribution by borough

In [ ]:
plt.figure(figsize=(8, 5))

for borough in ["Bronx", "Manhattan"]:
    group = data[data["Borough"] == borough]

    plt.hist(
        group["MeanCommute"],
        bins=20,
        alpha=0.55,
        label=borough
    )

plt.xlabel("Mean Commute Time (Minutes)")
plt.ylabel("Number of Census Tracts")
plt.title("Commute-Time Distribution by Borough")
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_commute_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 2. Public-transit use vs. mean commute time

In [ ]:
plt.figure(figsize=(8, 6))

for borough in ["Bronx", "Manhattan"]:
    group = data[data["Borough"] == borough]

    plt.scatter(
        group["Transit"],
        group["MeanCommute"],
        alpha=0.55,
        label=borough
    )

    slope, intercept = np.polyfit(
        group["Transit"],
        group["MeanCommute"],
        1
    )

    x_line = np.linspace(
        group["Transit"].min(),
        group["Transit"].max(),
        100
    )

    plt.plot(
        x_line,
        slope * x_line + intercept
    )

plt.xlabel("Workers Using Public Transit (%)")
plt.ylabel("Mean Commute Time (Minutes)")
plt.title("Public Transit Use and Commute Time by Borough")
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_transit_vs_commute.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 3. Bronx commute gap before and after controls

In [ ]:
bronx_compare = pd.DataFrame({
    "Model": [
        "Borough only",
        "Full controls"
    ],
    "Coefficient": [
        11.8335,
        9.1734
    ],
    "CI_Low": [
        10.917,
        8.150
    ],
    "CI_High": [
        12.750,
        10.197
    ]
})

errors = np.vstack([
    bronx_compare["Coefficient"] - bronx_compare["CI_Low"],
    bronx_compare["CI_High"] - bronx_compare["Coefficient"]
])

plt.figure(figsize=(7, 5))

plt.errorbar(
    bronx_compare["Model"],
    bronx_compare["Coefficient"],
    yerr=errors,
    fmt="o",
    capsize=5
)

plt.ylabel("Estimated Bronx Commute Gap (Minutes)")
plt.title("Bronx Commute Gap Before and After Controls")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_bronx_gap_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 4. Full OLS coefficient plot

In [ ]:
plot_data = (
    full_ols_results[
        full_ols_results["Variable"] != "const"
    ]
    .copy()
    .sort_values("Coefficient")
)

errors = np.vstack([
    plot_data["Coefficient"] - plot_data["CI_Low"],
    plot_data["CI_High"] - plot_data["Coefficient"]
])

plt.figure(figsize=(8, 5))

plt.errorbar(
    plot_data["Coefficient"],
    plot_data["Variable"],
    xerr=errors,
    fmt="o",
    capsize=4
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel("OLS Coefficient")
plt.ylabel("")
plt.title("Factors Associated with Mean Commute Time")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_ols_coefficients.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 5. Final descriptive statistics table

In [ ]:
final_summary = data.groupby("Borough").agg(
    Census_Tracts=("MeanCommute", "size"),
    Mean_Commute=("MeanCommute", "mean"),
    Public_Transit=("Transit", "mean"),
    Median_Income=("Income", "median"),
    Poverty=("Poverty", "mean"),
    Unemployment=("Unemployment", "mean"),
    Work_At_Home=("WorkAtHome", "mean")
).round(2)

final_summary


In [ ]:
final_summary.to_csv(
    OUTPUT_DIR / "final_descriptive_statistics.csv"
)

print("Final descriptive table saved.")


## 6. Output summary

In [ ]:
print("QSS 45 Final Figures and Tables")
print("--------------------------------")

print("1. final_commute_distribution.png")
print("2. final_transit_vs_commute.png")
print("3. final_bronx_gap_comparison.png")
print("4. final_ols_coefficients.png")
print("5. final_descriptive_statistics.csv")

print("\nAll files are saved in:", OUTPUT_DIR)
